### Phase 1: Data Understanding

This phase focuses on analyzing and understanding the datasets used for causal inference. The goal is to examine the structure, features, and domain-specific properties of the data before applying any modeling techniques.

The project uses three datasets from different domains:
- Education (student performance)
- Healthcare (diabetes dataset)
- Marketing (email campaign dataset)

Each dataset contains a treatment variable and an outcome variable, which are used to study cause-effect relationships.

Unlike traditional preprocessing-heavy approaches, this project preserves the natural structure of the datasets to maintain real-world authenticity. No artificial normalization or forced transformations are applied at this stage.

This phase ensures that the datasets are properly understood before moving to causal modeling.

In [11]:
import pandas as pd
import numpy as np
import os
import pickle
print("Libraries loaded successfully")

Libraries loaded successfully


In [12]:
ROOT_DIR = os.path.abspath("..")
DATA_PATH = os.path.join(ROOT_DIR, "data", "raw")
print("Root Directory:", ROOT_DIR)
print("Data Path:", DATA_PATH)

Root Directory: D:\UCSCR-UQ
Data Path: D:\UCSCR-UQ\data\raw


In [13]:
student_df = pd.read_csv(os.path.join(DATA_PATH, "student-mat.csv"), sep = ";")
student_df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


### Dataset Description

The datasets are loaded from their respective sources and represent real-world scenarios from different domains.

- The education dataset focuses on student attributes and performance indicators.
- The healthcare dataset includes medical attributes such as glucose levels and health outcomes.
- The marketing dataset contains information about email campaigns and customer responses.

Each dataset includes:
- A **treatment variable** (independent variable)
- An **outcome variable** (dependent variable)

These variables are later used for causal analysis.

### Initial Data Inspection

The initial rows of the dataset provide a quick overview of the data structure, feature types, and possible relationships.

This step helps in:
- Understanding feature distribution
- Identifying categorical and numerical variables
- Observing any irregularities or missing values

No modifications are made at this stage, as the objective is purely observational.

In [14]:
student_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

### Data Structure Analysis

The dataset contains a mix of numerical and categorical features. Understanding data types is essential for selecting appropriate preprocessing techniques in later phases.

Categorical variables will be handled carefully without losing their semantic meaning, and numerical variables will be used directly without unnecessary scaling.

This step ensures that the dataset structure is clearly understood before applying transformations.

In [15]:
student_df.isnull().sum()

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
G3            0
dtype: int64

In [16]:
student_df.describe()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,16.696203,2.749367,2.521519,1.448101,2.035443,0.334177,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861,10.908861,10.713924,10.415190
std,1.276043,1.094735,1.088201,0.697505,0.839240,0.743651,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096,3.319195,3.761505,4.581443
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.000000,0.000000,0.000000
25%,16.000000,2.000000,2.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,8.000000,9.000000,8.000000
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,11.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000,19.000000,19.000000,20.000000


In [17]:
SCHEMA = {"treatment" : "studytime", "outcome" : "G3", "confounders" : ["absences", "health", "failures"], "features" : ["studytime", "absences", "health", "failures"]}
print("Schema Defined Successfully")
SCHEMA

Schema Defined Successfully


{'treatment': 'studytime',
 'outcome': 'G3',
 'confounders': ['absences', 'health', 'failures'],
 'features': ['studytime', 'absences', 'health', 'failures']}

In [18]:
student_df[SCHEMA["features"]].head()

,studytime,absences,health,failures
0,2,6,3,0
1,2,4,3,0
2,2,10,3,3
3,3,2,5,0
4,2,4,5,0


In [19]:
with open("../artifacts/phase1/data_schema.pkl", "wb") as f:
    pickle.dump(SCHEMA, f)

### Conclusion

The data understanding phase confirms that the datasets are suitable for causal analysis and contain meaningful treatment and outcome variables.

By preserving the natural structure of the data, this project ensures that future analysis reflects realistic behavior rather than artificially optimized results.

This phase lays the foundation for preprocessing and causal modeling in the next stages.